# Module 5: Kernel Design for Commodity Forecasting

## Learning Objectives
By completing this notebook, you will:
1. Design composite kernels by combining basic kernels (addition, multiplication)
2. Implement periodic kernels for seasonal commodity patterns
3. Build multi-scale kernels capturing short, medium, and long-term dynamics
4. Apply domain knowledge to kernel construction for natural gas and agriculture
5. Compare kernel specifications using model selection criteria

## Prerequisites
- GP fundamentals from previous notebook
- Understanding of commodity seasonality
- Kernel functions (RBF, Matérn, Periodic)

## Estimated Time: 80 minutes

---

In [ ]:
learning_objectives(["Design composite kernels by combining basic kernels (addition, multiplication)", "Implement periodic kernels for seasonal commodity patterns", "Build multi-scale kernels capturing short, medium, and long-term dynamics", "Apply domain knowledge to kernel construction for natural gas and agriculture", "Compare kernel specifications using model selection criteria", "GP fundamentals from previous notebook"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats

np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")

## Conceptual Introduction

### The Kernel Zoo

**Key Insight:** Real commodity prices exhibit multiple timescale patterns. A single kernel is often insufficient. We need to **compose kernels** to capture:

1. **Trend** - Long-term supply/demand shifts
2. **Seasonality** - Weather, storage cycles, refinery maintenance
3. **Short-term noise** - Daily trading volatility
4. **Regime changes** - Policy shifts, structural breaks

### Kernel Algebra

**Addition:** $k_1(x, x') + k_2(x, x')$
- Combines independent sources of variation
- Example: Trend + Seasonal

**Multiplication:** $k_1(x, x') \times k_2(x, x')$
- Modulates one pattern by another
- Example: Seasonal amplitude that changes over time

**Scaling:** $\sigma^2 \cdot k(x, x')$
- Controls overall variance

### Example: Natural Gas Price Kernel

$$k_{\text{gas}}(t, t') = k_{\text{trend}}(t, t') + k_{\text{seasonal}}(t, t') \times k_{\text{modulation}}(t, t') + k_{\text{noise}}(t, t')$$

Where:
- Trend: RBF with large length scale (multi-year)
- Seasonal: Periodic (annual)
- Modulation: RBF (seasonal amplitude varies)
- Noise: White noise

## Part 1: Periodic Kernel for Seasonality

In [ ]:
# Purpose: Generate natural gas prices with strong seasonality
# Key Concept: Heating demand in winter, cooling demand in summer

n_days = 3 * 365  # 3 years
t = np.arange(n_days)

# Components
base_price = 3.0  # $/MMBtu

# Strong winter peaks (heating demand)
# Peak in Jan (day ~15) and trough in July (day ~195)
day_of_year = t % 365
seasonal = 2.0 * np.cos(2 * np.pi * (day_of_year - 15) / 365)

# Long-term trend (rising prices)
trend = 0.001 * t

# Noise
noise = np.random.normal(0, 0.3, n_days)

# Combine
gas_prices = base_price + seasonal + trend + noise

# Subsample for modeling (weekly observations)
obs_idx = np.arange(0, n_days, 7)
X_gas = t[obs_idx, None].astype(float)
y_gas = gas_prices[obs_idx]

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Time series
ax = axes[0]
ax.plot(t, gas_prices, alpha=0.5, linewidth=1, label='Daily Prices')
ax.scatter(X_gas, y_gas, c='red', s=30, alpha=0.7, label='Weekly Observations', zorder=5)
ax.set_xlabel('Days')
ax.set_ylabel('Price ($/MMBtu)')
ax.set_title('Natural Gas Prices with Annual Seasonality', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Seasonal pattern (averaged by day of year)
ax = axes[1]
seasonal_avg = np.array([gas_prices[day_of_year == d].mean() for d in range(365)])
ax.plot(range(365), seasonal_avg, linewidth=3, color='darkblue')
ax.axvline(15, color='red', linestyle='--', alpha=0.7, label='Winter Peak')
ax.axvline(195, color='green', linestyle='--', alpha=0.7, label='Summer Trough')
ax.set_xlabel('Day of Year')
ax.set_ylabel('Average Price ($/MMBtu)')
ax.set_title('Average Seasonal Pattern', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Data: {len(y_gas)} weekly observations over {n_days/365:.1f} years")
print(f"Price range: ${gas_prices.min():.2f} - ${gas_prices.max():.2f}")
print(f"Seasonal amplitude: ~${2*2.0:.2f}")

In [ ]:
# Purpose: Build GP with periodic kernel
# Key Concept: Periodic kernel captures repeating patterns

with pm.Model() as periodic_model:
    # Periodic kernel parameters
    period = pm.Normal('period', mu=365, sigma=10)  # Annual cycle
    ls_periodic = pm.Gamma('ls_periodic', alpha=2, beta=0.1)  # Smoothness within period
    sigma_periodic = pm.HalfNormal('sigma_periodic', sigma=3)  # Seasonal amplitude
    
    # Trend parameters
    ls_trend = pm.Gamma('ls_trend', alpha=2, beta=0.001)  # Long length scale
    sigma_trend = pm.HalfNormal('sigma_trend', sigma=2)
    
    # Noise
    sigma_n = pm.HalfNormal('sigma_n', sigma=0.5)
    
    # Composite kernel: Trend + Seasonality
    k_trend = sigma_trend**2 * pm.gp.cov.ExpQuad(1, ls=ls_trend)
    k_periodic = sigma_periodic**2 * pm.gp.cov.Periodic(1, period=period, ls=ls_periodic)
    
    k_total = k_trend + k_periodic
    
    # GP
    gp = pm.gp.Marginal(cov_func=k_total)
    
    # Likelihood
    y_obs = gp.marginal_likelihood('y_obs', X=X_gas, y=y_gas, sigma=sigma_n)
    
    # Sample
    trace_periodic = pm.sample(
        1000,
        tune=1500,
        target_accept=0.95,
        return_inferencedata=True,
        random_seed=42
    )

print("\nPosterior summary:")
print(az.summary(
    trace_periodic,
    var_names=['period', 'ls_periodic', 'sigma_periodic', 'sigma_trend'],
    round_to=2
))

In [ ]:
# Purpose: Make predictions and visualize decomposition
# Key Concept: Separate trend and seasonal components

X_pred = np.linspace(0, n_days + 180, 500)[:, None]

with periodic_model:
    f_pred = gp.conditional('f_pred', X_pred)
    ppc_periodic = pm.sample_posterior_predictive(
        trace_periodic,
        var_names=['f_pred'],
        random_seed=42
    )

f_mean = ppc_periodic.posterior_predictive['f_pred'].mean(dim=['chain', 'draw']).values
f_lower = np.percentile(ppc_periodic.posterior_predictive['f_pred'].values, 2.5, axis=(0, 1))
f_upper = np.percentile(ppc_periodic.posterior_predictive['f_pred'].values, 97.5, axis=(0, 1))

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

# Observations
ax.scatter(X_gas, y_gas, c='red', s=20, alpha=0.6, label='Observed', zorder=5)

# GP prediction
ax.plot(X_pred, f_mean, 'b-', linewidth=2, label='GP Mean (Trend + Seasonal)')
ax.fill_between(X_pred.flatten(), f_lower, f_upper, alpha=0.3, color='blue', label='95% CI')

# Forecast region
ax.axvspan(n_days, X_pred.max(), alpha=0.1, color='orange', label='Forecast Period')

ax.set_xlabel('Days', fontsize=12)
ax.set_ylabel('Price ($/MMBtu)', fontsize=12)
ax.set_title('GP with Periodic Kernel: Natural Gas Forecast', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

period_est = trace_periodic.posterior['period'].mean().item()
print(f"\nLearned period: {period_est:.1f} days (true: 365)")
print(f"GP successfully captures annual seasonality and forecasts into future")

## Exercise 5.3: Multi-Scale Kernel Design

**Task:** Design a kernel for crude oil prices that captures THREE timescales:

1. **Long-term trend** (multi-year): RBF with ℓ ~ 500-1000 days
2. **Medium-term cycles** (business cycle, ~2 years): Periodic or RBF with ℓ ~ 200-400 days  
3. **Short-term volatility** (weekly): Matérn-3/2 with ℓ ~ 10-30 days

Combine these using addition:

$$k_{\text{oil}} = k_{\text{long}} + k_{\text{medium}} + k_{\text{short}}$$

**Expected Output:**
- Model captures smooth trends + medium cycles + short-term noise
- Forecast uncertainty increases appropriately
- Hyperparameters learn correct timescales

**Hints:**
<details>
<summary>Hint 1</summary>
Use informative priors: pm.Gamma('ls_long', alpha=2, beta=0.002) for ℓ ~ 1000
</details>

<details>
<summary>Hint 2</summary>
Generate synthetic oil data with these components first
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Generate multi-scale oil price data
# 2. Build GP with three-component kernel
# 3. Fit and forecast

# Generate data
n_oil = 1000
t_oil = np.arange(n_oil)

# YOUR CODE: Create components and combine

with pm.Model() as multiscale_model:
    # YOUR CODE: Define composite kernel
    pass

your_answer = None  # Replace with your trace

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_5_3():
    assert your_answer is not None, "Implement your solution!"
    assert isinstance(your_answer, az.InferenceData), "Return InferenceData"
    
    # Check for multiple length scales
    var_names = list(your_answer.posterior.data_vars.keys())
    ls_params = [v for v in var_names if 'ls' in v.lower()]
    
    assert len(ls_params) >= 2, "Need at least 2 different length scales"
    
    print("Exercise 5.3 passed!")
    print(f"Found {len(ls_params)} length scale parameters")

test_exercise_5_3()

## Part 2: Kernel Multiplication (Locally Periodic)

**Use case:** Seasonal amplitude that changes over time.

Example: Agricultural prices have strong harvest seasonality, but the amplitude of seasonality varies with supply shocks.

In [ ]:
# Purpose: Generate corn prices with time-varying seasonal amplitude
# Key Concept: Harvest impact varies by year (drought, bumper crop)

n_years = 5
n_corn = n_years * 12  # Monthly
t_corn = np.arange(n_corn)

# Base price
base = 4.0

# Seasonal component (harvest in Sep-Oct, months 9-10)
month_of_year = t_corn % 12
seasonal_base = -0.5 * ((month_of_year == 9) | (month_of_year == 10)).astype(float)

# Time-varying amplitude (some years have bigger harvest drops)
amplitude_variation = 1.0 + 0.5 * np.sin(2 * np.pi * t_corn / 60)  # Varies over time

# Combine
seasonal_varying = seasonal_base * amplitude_variation
corn_prices = base + seasonal_varying + np.random.normal(0, 0.15, n_corn)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

ax = axes[0]
ax.plot(t_corn, corn_prices, 'o-', linewidth=2, markersize=6)
ax.set_xlabel('Month')
ax.set_ylabel('Price ($/bushel)')
ax.set_title('Corn Prices with Time-Varying Seasonal Amplitude', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(t_corn, amplitude_variation, linewidth=3, color='darkgreen')
ax.set_xlabel('Month')
ax.set_ylabel('Seasonal Amplitude Multiplier')
ax.set_title('How Seasonal Effect Strength Varies Over Time', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(1, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("Note: Harvest drops (Sep-Oct) have varying magnitudes across years")

In [ ]:
# Purpose: Model with kernel multiplication
# Key Concept: k_periodic × k_rbf captures locally periodic behavior

X_corn = t_corn[:, None].astype(float)
y_corn = corn_prices

with pm.Model() as locally_periodic_model:
    # Periodic kernel (annual seasonality)
    period_corn = pm.Normal('period_corn', mu=12, sigma=1)
    ls_periodic = pm.Gamma('ls_periodic', alpha=2, beta=1)
    sigma_periodic = pm.HalfNormal('sigma_periodic', sigma=1)
    
    # Modulation kernel (how seasonal amplitude varies)
    ls_modulation = pm.Gamma('ls_modulation', alpha=2, beta=0.1)  # Slower variation
    sigma_modulation = pm.HalfNormal('sigma_modulation', sigma=1)
    
    # Base level
    sigma_base = pm.HalfNormal('sigma_base', sigma=1)
    ls_base = pm.Gamma('ls_base', alpha=2, beta=0.05)
    
    # Noise
    sigma_n = pm.HalfNormal('sigma_n', sigma=0.3)
    
    # Composite kernel: Base + (Periodic × Modulation)
    k_base = sigma_base**2 * pm.gp.cov.ExpQuad(1, ls=ls_base)
    k_periodic = sigma_periodic**2 * pm.gp.cov.Periodic(1, period=period_corn, ls=ls_periodic)
    k_modulation = sigma_modulation**2 * pm.gp.cov.ExpQuad(1, ls=ls_modulation)
    
    k_total = k_base + k_periodic * k_modulation
    
    # GP
    gp = pm.gp.Marginal(cov_func=k_total)
    y_obs = gp.marginal_likelihood('y_obs', X=X_corn, y=y_corn, sigma=sigma_n)
    
    # Sample
    trace_corn = pm.sample(
        1000,
        tune=1500,
        target_accept=0.95,
        return_inferencedata=True,
        random_seed=42
    )

print(az.summary(trace_corn, var_names=['period_corn', 'ls_modulation'], round_to=2))

In [ ]:
# Purpose: Forecast with locally periodic kernel

X_corn_pred = np.arange(n_corn + 12)[:, None].astype(float)

with locally_periodic_model:
    f_pred_corn = gp.conditional('f_pred', X_corn_pred)
    ppc_corn = pm.sample_posterior_predictive(
        trace_corn,
        var_names=['f_pred'],
        random_seed=42
    )

f_mean_corn = ppc_corn.posterior_predictive['f_pred'].mean(dim=['chain', 'draw']).values
f_std_corn = ppc_corn.posterior_predictive['f_pred'].std(dim=['chain', 'draw']).values

plt.figure(figsize=(14, 6))
plt.scatter(X_corn, y_corn, c='darkgreen', s=50, alpha=0.7, label='Observed', zorder=5)
plt.plot(X_corn_pred, f_mean_corn, 'b-', linewidth=2, label='GP Mean')
plt.fill_between(
    X_corn_pred.flatten(),
    f_mean_corn - 2*f_std_corn,
    f_mean_corn + 2*f_std_corn,
    alpha=0.3,
    label='95% CI'
)
plt.axvspan(n_corn, n_corn+12, alpha=0.1, color='orange', label='Forecast')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Price ($/bushel)', fontsize=12)
plt.title('Corn Prices: Locally Periodic GP (Kernel Multiplication)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Model captures time-varying seasonal amplitude via kernel multiplication")

## Summary

### Key Takeaways

1. **Kernel composition** - Addition combines independent patterns, multiplication modulates amplitude

2. **Periodic kernels** - Essential for seasonal commodities (natural gas, agriculture)

3. **Multi-scale modeling** - Real prices need multiple timescale components

4. **Domain knowledge** - Kernel design encodes market structure (heating seasons, harvest timing)

5. **Model comparison** - Use WAIC/LOO to select kernel specifications

### Common Kernel Patterns for Commodities

| Commodity | Kernel Structure |
|-----------|------------------|
| Natural Gas | Trend + Periodic(365) × RBF |
| Crude Oil | RBF(long) + RBF(medium) + Matérn(short) |
| Corn | Base + Periodic(12) × RBF |
| Electricity | Periodic(365) + Periodic(7) + WhiteNoise |

### What's Next

**Module 6: Advanced Inference** - Efficient MCMC sampling for GP models

**Module 8: Fundamentals Integration** - Combine GPs with fundamental covariates (stocks, weather)

---

*"The kernel is the soul of the GP - design it to match the physics of your market."*